# Prediksi Rating Pengunjung Destinasi Wisata di India Menggunakan Machine Learning

**Metode:** Regresi  
**Model:** Linear Regression dan Random Forest Regression  
**Target:** `user_rating`

Notebook ini disusun mengikuti tahapan proyek akhir Praktikum Data Science 2026/2027. Seluruh fitur model dibatasi pada variabel numerik agar sesuai dengan materi praktikum yang telah dipelajari.

> **Catatan penting:** Hasil analisis dan evaluasi baru dapat ditulis setelah seluruh sel dijalankan. Jangan mengisi atau mengarang nilai metrik secara manual.

## 1. Business Understanding

### 1.1 Latar Belakang dan Pernyataan Masalah

Destinasi wisata menerima pengunjung dengan karakteristik kunjungan dan kondisi lingkungan yang beragam. Kondisi seperti suhu, kelembapan, curah hujan, tingkat keramaian, waktu kunjungan, dan lokasi berpotensi berkaitan dengan penilaian pengunjung terhadap pengalaman wisatanya.

Pengelola destinasi membutuhkan informasi berbasis data untuk memahami karakteristik kunjungan dan memperkirakan rating pengunjung. Proyek ini menggunakan pendekatan **regresi** untuk memprediksi `user_rating` berdasarkan fitur numerik yang tersedia pada dataset.

Analisis ini bersifat prediktif dan eksploratif. Hasilnya tidak langsung membuktikan hubungan sebab-akibat antara suatu kondisi dan rating.
### 1.2 Rumusan Masalah

1. Bagaimana karakteristik dan kualitas data kunjungan wisata pada dataset?
2. Bagaimana distribusi rating pengunjung (`user_rating`)?
3. Seberapa baik fitur numerik yang tersedia dapat digunakan untuk memprediksi rating pengunjung?
4. Di antara Linear Regression dan Random Forest Regression, model mana yang menghasilkan prediksi terbaik berdasarkan MAE, RMSE, dan R²?

### 1.3 Tujuan

1. Mengidentifikasi karakteristik, distribusi, dan kualitas dataset.
2. Menganalisis distribusi rating pengunjung dan hubungan awalnya dengan fitur numerik.
3. Membangun model prediksi rating menggunakan Linear Regression dan Random Forest Regression.
4. Membandingkan kedua model menggunakan MAE, RMSE, dan R² serta menentukan model terbaik berdasarkan hasil pengujian.

### 1.4 Stakeholder dan Manfaat

- **Pengelola destinasi wisata:** memperoleh gambaran awal faktor kunjungan yang berkaitan dengan rating.
- **Calon pengunjung:** secara tidak langsung dapat memperoleh wawasan mengenai kondisi kunjungan, meskipun model ini bukan sistem rekomendasi destinasi.
- **Peneliti/mahasiswa:** memahami penerapan tahapan data science dan regresi pada data pariwisata.

### 1.5 Tujuan Terukur

Proyek menghasilkan dua model regresi yang diuji pada data uji yang dipisahkan berdasarkan destinasi (`place_id`). Kinerja model dilaporkan menggunakan MAE, RMSE, dan R². Model dengan kesalahan prediksi lebih rendah dan R² lebih baik dipertimbangkan sebagai model yang lebih sesuai, dengan tetap memperhatikan keterbatasan data.

### 1.6 Batasan dan Asumsi

- Dataset berisi catatan kunjungan wisata, bukan satu baris unik untuk setiap destinasi.
- Target penelitian adalah rating numerik `user_rating`.
- Fitur model hanya menggunakan variabel numerik yang dipilih berdasarkan relevansi dan pemeriksaan potensi kebocoran data.
- Kolom teks dan kategori tidak diubah menjadi angka karena transformasi teks/kategori belum menjadi fokus materi praktikum.
- Evaluasi menggunakan pemisahan berdasarkan `place_id` agar catatan dari destinasi yang sama tidak tersebar pada data latih dan data uji.
- Hasil model bergantung pada kualitas, representativitas, dan keaslian dataset.

## 2. Data Gathering

### 2.1 Sumber Dataset

Nama dataset: **Indian Tourist Attraction Dataset**  
Sumber: Kaggle  
Tautan: https://www.kaggle.com/datasets/dakshineswarm/indian-tourist-attraction-dataset

Dataset disertakan dalam repository dengan nama `indian_tourist_places_dataset.csv`.

**Pemeriksaan kredibilitas:** sebelum pengumpulan, kelompok perlu memeriksa halaman sumber untuk memastikan deskripsi dataset, asal data, lisensi, dan keterangan apakah data merupakan data observasi atau data sintetis. Keberadaan dataset di Kaggle saja belum cukup untuk membuktikan keaslian data. Tuliskan hasil verifikasi sumber tersebut di sini setelah diperiksa.
### 2.2 Memuat Dataset

Jalankan notebook dari folder yang sama dengan file CSV. Jika file berada di lokasi lain, ubah nilai `DATA_PATH`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


DATA_PATH = "indian_tourist_places_dataset.csv"

df = pd.read_csv(DATA_PATH)
print("Ukuran dataset:", df.shape)
display(df.head())

### 2.3 Gambaran Dataset dan Arti Kolom

Setiap baris merepresentasikan satu catatan kunjungan/pengamatan. Dataset memuat identitas destinasi, lokasi, waktu kunjungan, cuaca, rating, ulasan, dan informasi rekomendasi.

Tabel berikut menampilkan nama kolom, tipe data, dan jumlah nilai kosong. Penjelasan rinci kolom dapat disesuaikan dengan dokumentasi asli dari halaman dataset.

### 2.4 Alasan Pemilihan Dataset

Dataset dipilih karena memiliki lebih dari 100 baris dan menyediakan sejumlah variabel numerik yang dapat dieksplorasi sebagai prediktor rating. Dataset juga memiliki nilai kosong sehingga tahap cleaning dan preprocessing dapat dilakukan berdasarkan kondisi aktual data.

**Catatan:** syarat jumlah baris/fitur dan adanya missing value dapat diperiksa langsung. Kredibilitas, lisensi, dan status keaslian tetap harus diverifikasi dari sumber dataset.

## 3. Exploratory Data Analysis (EDA)

EDA dilakukan untuk memahami struktur data, distribusi target, nilai kosong, duplikasi, dan hubungan awal antara fitur numerik dengan rating. Interpretasi di bawah setiap visual harus disesuaikan kembali setelah grafik ditampilkan.

### 3.1 Pemeriksaan Missing Value dan Duplikasi

**Interpretasi:** nilai kosong perlu diperiksa per kolom. Baris tanpa target `user_rating` tidak dapat digunakan untuk pelatihan model supervised regression. Missing pada fitur dapat ditangani menggunakan imputasi yang dipelajari hanya dari data latih agar tidak terjadi kebocoran informasi.

### 3.2 Distribusi Rating Pengunjung

**Interpretasi:** perhatikan apakah rating terkonsentrasi pada nilai tertentu, apakah distribusinya miring, serta rentang minimum dan maksimum. Distribusi yang sangat terkonsentrasi dapat membuat prediksi mendekati rata-rata terlihat baik; karena itu model perlu dibandingkan dengan baseline sederhana.

### 3.3 Distribusi Fitur Numerik

**Interpretasi:** gunakan histogram untuk mengenali kemencengan distribusi, rentang yang tidak wajar, dan kemungkinan outlier. Jangan menghapus outlier hanya karena nilainya ekstrem; pastikan ada alasan substantif.

### 3.4 Korelasi Awal dengan Target

**Interpretasi:** korelasi menunjukkan hubungan linear awal, bukan sebab-akibat. Korelasi yang rendah tidak otomatis berarti fitur tidak berguna bagi Random Forest, karena model pohon dapat menangkap pola non-linear. Periksa juga apakah fitur berpotensi dibentuk dari target.

## 4. Cleaning dan Preprocessing

### 4.1 Menentukan Target dan Fitur

Target adalah `user_rating`. Fitur awal dipilih dari variabel numerik yang masuk akal secara konteks. Kolom ID (`row_id`, `place_id`) tidak digunakan sebagai prediktor. Kolom yang berpotensi menjadi kebocoran target, seperti `sentiment_score` dan `final_recommendation_score`, dikeluarkan karena definisi/formula pembentukannya tidak dipastikan dan dapat berkaitan langsung dengan penilaian/rekomendasi.

Kolom `sentiment_label`, `review_text`, `would_revisit`, dan `would_recommend` tidak digunakan karena bukan fitur numerik yang dibahas dalam rancangan ini.

### 4.2 Menangani Target Kosong dan Duplikasi

Baris yang targetnya kosong dikeluarkan karena nilai rating yang hendak diprediksi tidak tersedia. Duplikasi identik diperiksa dan dihapus jika ditemukan. Missing value pada fitur tidak diisi sebelum pemisahan data; imputasi dilakukan di dalam pipeline setelah data dibagi.

### 4.3 Pemeriksaan Rentang Target dan Fitur

Periksa nilai minimum dan maksimum untuk memastikan rentang masuk akal berdasarkan dokumentasi dataset. Jika ditemukan nilai yang tidak valid, dokumentasikan alasan dan aturan penanganannya. Jangan menetapkan batas rating tanpa merujuk definisi skala rating pada sumber data.

### 4.4 Pemisahan Data Berdasarkan Destinasi

Dataset memiliki banyak catatan untuk destinasi yang sama. Pemisahan acak per baris berisiko menempatkan catatan dari destinasi yang sama ke data latih dan data uji. Untuk menguji kemampuan generalisasi ke destinasi yang tidak terlihat saat pelatihan, digunakan `GroupShuffleSplit` dengan `place_id` sebagai kelompok.

Jika tujuan penelitian sebenarnya hanya memprediksi kunjungan baru pada destinasi yang sudah pernah diamati, strategi split dapat berbeda dan harus dijelaskan.

### 4.5 Imputasi dan Standardisasi

Nilai kosong pada fitur numerik diisi menggunakan median dari data latih. Standardisasi digunakan pada Linear Regression sebagai bagian dari pipeline agar skala fitur lebih sebanding. Random Forest tidak memerlukan standardisasi, tetapi tetap menggunakan imputasi median. Pipeline memastikan proses preprocessing dipelajari dari data latih saja.

## 5. Modelling

Dua model dari materi praktikum dibandingkan:

1. **Linear Regression:** model dasar yang mempelajari hubungan linear antara fitur dan target. Model ini relatif mudah diinterpretasikan.
2. **Random Forest Regression:** gabungan beberapa decision tree yang dapat menangkap hubungan non-linear dan interaksi fitur.

Kedua model menggunakan data latih dan diuji pada data uji yang sama agar perbandingan adil.

## 6. Evaluation

### 6.1 Metrik Evaluasi

- **MAE (Mean Absolute Error):** rata-rata besar kesalahan absolut prediksi. Jika MAE = 0,30, secara rata-rata prediksi meleset sekitar 0,30 poin rating.
- **RMSE (Root Mean Squared Error):** akar rata-rata kuadrat kesalahan; lebih sensitif terhadap kesalahan besar.
- **R²:** proporsi variasi target yang dapat dijelaskan model pada data uji. Nilai dapat negatif jika model lebih buruk daripada prediksi rata-rata sederhana.

Model yang baik diharapkan memiliki MAE dan RMSE rendah serta R² lebih tinggi. Interpretasi tetap mempertimbangkan distribusi target dan baseline.

### 6.2 Visualisasi Aktual vs Prediksi

### 6.4 Distribusi Residual

### 6.5 Penentuan Model Terbaik

Model terbaik ditentukan dari hasil aktual pada tabel metrik, bukan ditentukan sebelumnya. Prioritaskan MAE yang lebih rendah karena mudah diterjemahkan ke satuan poin rating; gunakan RMSE dan R² sebagai pertimbangan tambahan. Jika hasil metrik saling bertentangan, jelaskan trade-off tersebut.